In [1]:
"""
Copyright (c) 2026 Leah Ding's Research Group, American University

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at:

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.

Funding Acknowledgement
-----------------------
This work was supported by the U.S. National Aeronautics and Space Administration (NASA)
and the National Science Foundation (NSF). Any opinions, findings, conclusions, or
recommendations expressed in this material are those of the authors and do not necessarily
reflect the views of NASA or the NSF.


Interactive Mask Verification Tool
==================================
Usage in Jupyter Notebook:
  1. Run all cells
  2. Use "← Prev" / "Next →" buttons to browse samples
  3. Click on the mask grid to modify the mask
  4. Click "✓ Verify & Save" to save verification results and modified mask
  5. Click "✗ Un-verify" to cancel verification
  6. Progress is saved in real-time to the result folder

"""

import os
import csv
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import ipywidgets as widgets
from IPython.display import display as ipy_display, clear_output



In [2]:
# ==================== Configuration ====================
Sat_DIR   = Path("./data/sat_dataset")
MASK_DIR  = Path("./data/all_mask")
RESULT_DIR = Path("./result")


CSV_PATH   = RESULT_DIR / "verification.csv"
MODIFIED_MASK_DIR = RESULT_DIR / "modified_mask"

MASK_SIZE = 32  # 32x32 grid
SHOW_GRID_LINES = True  # Show tic-tac-toe reference lines on plots and mask editor

# Channels to display (consistent with the first row of plot_sample_comparison)
DISPLAY_CHANNELS = [
    (2,  r"M10_rad ($W \cdot m^{-2} \cdot \mu m^{-1} \cdot sr^{-1}$)", (0, 0.5),   1.0),
    (9,  "BTI04 (K)",         (260, 310),  1.0),
    (11, r"DNB ($\text{nW} \cdot \text{cm}^{-2} \cdot \text{sr}^{-1}$)", (0, 150),    1e9),
]

CHANNEL_CAPS = {
    11: (150*1e-9, 150*1e-9),
}

def cap_channel(ch_idx, data):
    if ch_idx in CHANNEL_CAPS:
        threshold, cap_value = CHANNEL_CAPS[ch_idx]
        data = data.copy()
        data[data > threshold] = cap_value
    return data



In [3]:
# ==================== File Matching and Sorting ====================
sat_files  = sorted(Sat_DIR.glob("*.npy"))
mask_files = sorted(MASK_DIR.glob("*.png"))

sat_map  = {f.stem: f for f in sat_files}
mask_map = {f.stem.replace("_norm_Simple Segmentation", ""): f for f in mask_files}

common = sorted(set(sat_map.keys()) & set(mask_map.keys()))
print(f"Matched {len(common)} samples (sat: {len(sat_files)}, mask: {len(mask_files)})")



Matched 1906 samples (sat: 3255, mask: 1906)


In [4]:
# ==================== CSV Management ====================
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODIFIED_MASK_DIR.mkdir(parents=True, exist_ok=True)

def load_csv():
    """Load existing verification.csv, return dict {filename: quality}"""
    results = {}
    if CSV_PATH.exists():
        with open(CSV_PATH, "r", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                results[row["filename"]] = row["quality"] == "True"
    return results

def save_csv(results):
    """Save verification results to CSV"""
    with open(CSV_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["filename", "quality"])
        writer.writeheader()
        for fname in sorted(results.keys()):
            writer.writerow({"filename": fname, "quality": str(results[fname])})

def find_resume_index(results):
    """Find the last verified position, return the next unverified index"""
    if len(common) == 0:
        return 0
    for i, fname in enumerate(common):
        if fname not in results:
            return i
    return len(common) - 1   # All verified, return to the last one



In [5]:
# ==================== Interactive Mask Drawer ====================
class MaskEditor:
    """32x32 editable mask grid"""

    def __init__(self, size=MASK_SIZE, cell_size="10px"):
        self.size = size
        self.buttons = []
        self._mask_modified = False
        self._cell_size = cell_size

        self.brush_size = 1  # Logic: 1, 3, 5...
        self._updating = False  # Prevent recursive updates

        for i in range(size * size):
            btn = widgets.ToggleButton(
                value=False,
                description="",
                layout=widgets.Layout(width=cell_size, height=cell_size, padding="0", margin="0"),
                style={"button_width": "initial"},
            )
            # Store coordinate
            row, col = divmod(i, size)
            btn._coord = (row, col)

            btn.observe(self._on_toggle, names="value")
            self._update_color(btn)
            self.buttons.append(btn)

        self.grid = widgets.GridBox(
            children=self.buttons,
            layout=widgets.Layout(
                width="fit-content",
                grid_template_columns=f"repeat({size}, {cell_size})",
                grid_gap="0px",
            ),
        )

        # Apply tic-tac-toe reference line CSS on grid cells
        if SHOW_GRID_LINES:
            line1 = size // 3       # ~10
            line2 = size * 2 // 3   # ~21
            css_rules = []
            for i in range(size * size):
                row, col = divmod(i, size)
                borders = []
                if row == line1:
                    borders.append("border-top: 2px dashed rgba(255,0,0,0.6) !important")
                if row == line2:
                    borders.append("border-top: 2px dashed rgba(255,0,0,0.6) !important")
                if col == line1:
                    borders.append("border-left: 2px dashed rgba(255,0,0,0.6) !important")
                if col == line2:
                    borders.append("border-left: 2px dashed rgba(255,0,0,0.6) !important")
                if borders:
                    cls = f"grid-ref-{row}-{col}"
                    self.buttons[i].add_class(cls)
                    css_rules.append(f".{cls} {{ {'; '.join(borders)}; }}")
            self._grid_css = widgets.HTML(
                f"<style>{''.join(css_rules)}</style>"
            )
        else:
            self._grid_css = widgets.HTML("")

        # Cell size slider for resizing
        self.cell_size_slider = widgets.IntSlider(
            value=int(cell_size.replace("px", "")),
            min=7, max=32, step=1,
            description="Grid Size:",
            layout=widgets.Layout(width="400px"),
        )
        self.cell_size_slider.observe(self._on_resize, names="value")

    def _on_resize(self, change):
        """Dynamically adjust mask grid cell size"""
        new_size = f"{change['new']}px"
        self._cell_size = new_size
        for btn in self.buttons:
            btn.layout.width = new_size
            btn.layout.height = new_size
        self.grid.layout.grid_template_columns = f"repeat({self.size}, {new_size})"

    def _update_color(self, btn):
        btn.style.button_color = "lightgreen" if btn.value else "#555"

    def _on_toggle(self, change):
        if self._updating:
            return

        self._updating = True
        try:
            owner = change["owner"]
            new_val = change["new"]
            row_center, col_center = owner._coord

            # Update neighbors based on brush size
            r = self.brush_size // 2

            # Simple square brush
            for i in range(-r, r + 1):
                for j in range(-r, r + 1):
                    nr, nc = row_center + i, col_center + j
                    if 0 <= nr < self.size and 0 <= nc < self.size:
                        idx = nr * self.size + nc
                        btn = self.buttons[idx]
                        if btn.value != new_val:
                            btn.value = new_val
                            self._update_color(btn)

            self._update_color(owner)
            self._mask_modified = True
        finally:
            self._updating = False

    def set_mask(self, mask_array):
        """Load a 32x32 bool/int mask into the grid"""
        self._mask_modified = False
        self._updating = True  # Prevent triggering logic during load
        try:
            flat = mask_array.flatten().astype(bool)
            for i, btn in enumerate(self.buttons):
                btn.value = bool(flat[i])
                self._update_color(btn)
        finally:
            self._updating = False

    def get_mask(self):
        """Return a 32x32 0/1 numpy array"""
        vals = [int(btn.value) for btn in self.buttons]
        return np.array(vals).reshape(self.size, self.size)

    @property
    def modified(self):
        return self._mask_modified



In [6]:
# ==================== Main Verification UI ====================
class MaskVerificationTool:
    """Interactive Mask Verification Tool"""

    def __init__(self):
        # Load existing verification results
        self.results = load_csv()
        if len(common) == 0:
            raise RuntimeError(
                f"No files matched!\n"
                f"  sat_dir={Sat_DIR} ({len(sat_files)} files)\n"
                f"  mask_dir={MASK_DIR} ({len(mask_files)} files)\n"
                f"Please check if paths and filenames match."
            )
        self.current_idx = find_resume_index(self.results)

        # === UI 组件 ===
        # Navigation
        self.btn_prev = widgets.Button(description="← Prev", layout=widgets.Layout(width="100px"))
        self.btn_next = widgets.Button(description="Next →", layout=widgets.Layout(width="100px"))
        self.btn_verify = widgets.Button(
            description="✓ Verify & Save",
            button_style="success",
            layout=widgets.Layout(width="140px"),
        )
        self.btn_unverify = widgets.Button(
            description="✗ Undo",
            button_style="warning",
            layout=widgets.Layout(width="120px"),
        )
        self.btn_goto = widgets.Button(description="Go", layout=widgets.Layout(width="60px"))
        self.text_goto = widgets.IntText(
            value=self.current_idx,
            layout=widgets.Layout(width="80px"),
            description="",
        )

        # Status display
        self.label_info = widgets.HTML(value="")
        self.label_status = widgets.HTML(value="")

        # Plots output
        self.plot_output = widgets.Output()

        # Mask editor
        self.mask_editor = MaskEditor(size=MASK_SIZE)

        # Brush size slider
        self.slider_brush = widgets.IntSlider(
            value=1, min=1, max=5, step=2,
            description="Brush Size:",
            layout=widgets.Layout(width="250px")
        )
        self.slider_brush.observe(self._on_brush_change, names="value")

        # Wire up events
        self.btn_prev.on_click(self._on_prev)
        self.btn_next.on_click(self._on_next)
        self.btn_verify.on_click(self._on_verify)
        self.btn_unverify.on_click(self._on_unverify)
        self.btn_goto.on_click(self._on_goto)

        # Layout
        nav_bar = widgets.HBox([
            self.btn_prev,
            self.btn_next,
            widgets.Label("  "),
            self.btn_verify,
            self.btn_unverify,
            widgets.Label("  |  Go to:"),
            self.text_goto,
            self.btn_goto,
        ])

        # Top: channel plots | Bottom: mask editor + status
        self.container = widgets.VBox([
            self.label_info,
            nav_bar,
            self.plot_output,
            widgets.HBox([
                widgets.HTML("<b>Mask Editor</b>"),
                widgets.Label("   |   "),
                self.slider_brush,
                widgets.Label("   |   "),
                self.mask_editor.cell_size_slider,
            ]),
            self.mask_editor._grid_css,
            self.mask_editor.grid,
            self.label_status,
        ])

        # Initial load
        self._load_sample()

    def _update_info_bar(self):
        """Update only the top info bar (verification status, counts, etc.), do not redraw plots"""
        idx = self.current_idx
        fname = common[idx]

        verified_count = sum(1 for f in common if f in self.results)
        is_verified = fname in self.results
        status_text = "✅ Verified &nbsp;&nbsp;" if is_verified else "⬜ Unverified"

        bg_color = "#a5d6a7" if is_verified else "#f0f0f0"
        self.label_info.value = (
            f"<div style='font-size:14px; padding:5px; background:{bg_color}; border-radius:5px;'>"
            f"<b>Sample #{idx}</b> / {len(common)-1} &nbsp;&nbsp;|&nbsp;&nbsp; "
            f"<code>{fname}</code> &nbsp;&nbsp;|&nbsp;&nbsp; "
            f"{status_text} &nbsp;&nbsp;|&nbsp;&nbsp; "
            f"Verified: {verified_count}/{len(common)}"
            f"</div>"
        )
        self.text_goto.value = idx

    def _load_sample(self):
        """Load sample at current index and update UI"""
        idx = self.current_idx
        fname = common[idx]

        # Update info bar
        self._update_info_bar()

        # Load data
        sat = np.load(sat_map[fname])              # (17, 32, 32)
        mask_img = np.array(Image.open(mask_map[fname]))   # (32, 32), uint8

        # Check if modified mask exists; prefer loading it
        modified_path = MODIFIED_MASK_DIR / f"{fname}.png"
        is_modified = False
        if modified_path.exists():
            mask_img = np.array(Image.open(modified_path))
            is_modified = True

        # mask: 255 → True (fire), 0 → False (background)
        mask_bool = mask_img.astype(bool)

        # Load mask into editor (Note: inverted during original load to match analysis code, but editor displays real mask)
        self.mask_editor.set_mask(mask_bool)

        # Plot channel images — display skeleton screen first to prevent flickering
        with self.plot_output:
            clear_output(wait=True)
            ipy_display(widgets.HTML(
                "<div style='height:330px; display:flex; align-items:center; "
                "justify-content:center; background:#f5f5f5; border-radius:8px; "
                "color:#999; font-size:14px;'>⏳ Loading sample...</div>"
            ))

        with self.plot_output:
            clear_output(wait=True)
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            fig.suptitle(f"Sample #{idx}: {fname}", fontsize=12, fontweight="bold")

            for col, (ch_idx, ch_name, clim, scale) in enumerate(DISPLAY_CHANNELS):
                ax = axes[col]
                ch_data = cap_channel(ch_idx, sat[ch_idx])
                plot_data = ch_data * scale
                vmin, vmax = clim
                im = ax.imshow(plot_data, cmap="viridis", aspect="equal", vmin=vmin, vmax=vmax)
                # suffix = " (×1e9)" if scale != 1.0 else ""
                ax.set_title(f"{ch_name}", fontsize=10)
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                ax.set_xticks([])
                ax.set_yticks([])
                if SHOW_GRID_LINES:
                    for pos in [MASK_SIZE // 3, MASK_SIZE * 2 // 3]:
                        ax.axhline(y=pos - 0.5, color="white", linestyle="--", linewidth=0.8, alpha=0.6)
                        ax.axvline(x=pos - 0.5, color="white", linestyle="--", linewidth=0.8, alpha=0.6)

            # mask image
            ax_mask = axes[3]
            im_mask = ax_mask.imshow(1 - mask_bool.astype(float), cmap="gray", vmin=0, vmax=1, aspect="equal")
            cbar_mask = fig.colorbar(im_mask, ax=ax_mask, fraction=0.046, pad=0.04)
            cbar_mask.ax.set_visible(False)  # Occupy space but hide visually

            mask_title = "Modified Mask" if is_modified else "Original Mask"
            ax_mask.set_title(mask_title, fontsize=10)
            ax_mask.set_xticks([])
            ax_mask.set_yticks([])
            if SHOW_GRID_LINES:
                for pos in [MASK_SIZE // 3, MASK_SIZE * 2 // 3]:
                    ax_mask.axhline(y=pos - 0.5, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
                    ax_mask.axvline(x=pos - 0.5, color="red", linestyle="--", linewidth=0.8, alpha=0.6)

            plt.tight_layout()
            plt.show()

        self._update_status("")

    def _update_status(self, msg):
        color = "#2e7d32" if "Success" in msg or "✅" in msg else "#c62828" if "Error" in msg else "#555"
        self.label_status.value = f"<div style='color:{color}; padding:3px; font-size:12px;'>{msg}</div>"

    def _on_prev(self, _):
        if self.current_idx > 0:
            self.current_idx -= 1
            self._load_sample()

    def _on_next(self, _):
        if self.current_idx < len(common) - 1:
            self.current_idx += 1
            self._load_sample()

    def _on_goto(self, _):
        target = self.text_goto.value
        if 0 <= target < len(common):
            self.current_idx = target
            self._load_sample()
        else:
            self._update_status(f"Error: index {target} out of range [0, {len(common)-1}]")

    def _on_verify(self, _):
        """Verify current sample: save quality=True to CSV; if mask modified, save PNG"""
        fname = common[self.current_idx]

        # Save verification results
        self.results[fname] = True
        save_csv(self.results)

        # If mask was modified, save the modified mask and redraw plot
        if self.mask_editor.modified:
            new_mask = self.mask_editor.get_mask()   # 0/1 array
            mask_uint8 = (new_mask * 255).astype(np.uint8)
            save_path = MODIFIED_MASK_DIR / f"{fname}.png"
            Image.fromarray(mask_uint8).save(save_path)
            self._update_status(f"✅ Verified + mask saved to {save_path.name}")
            # Redraw plot so the mask image reflects the saved changes
            self._load_sample()
        else:
            self._update_status(f"✅ Verified (mask not modified)")
            # Refresh info bar only (no need to redraw plot)
            self._update_info_bar()

    def _on_unverify(self, _):
        """Un-verify current sample"""
        fname = common[self.current_idx]
        if fname in self.results:
            del self.results[fname]
            save_csv(self.results)
            self._update_status(f"❌ Verification cancelled")
        else:
            self._update_status(f"⚠️ Current sample not verified")
        self._update_info_bar()

    def display(self):
        ipy_display(self.container)

    def _on_brush_change(self, change):
        self.mask_editor.brush_size = change["new"]




In [7]:
# ==================== Start Tool ====================
tool = MaskVerificationTool()
tool.display()
